In [1]:
import time
import requests
import pandas as pd
import numpy as np

from functools import reduce
from linearmodels.panel import PanelOLS
from statsmodels.stats.multitest import multipletests

In [2]:
# Study period and climate-normal period

START_YEAR = 2005
END_YEAR = 2022

NORMAL_START = 1991
NORMAL_END = 2020

CLIMATE_HISTORY_START = 1901
CRU_END_YEAR = 2022

In [3]:
# Candidate economies, grouped using the study's income-group classification

lower_income_candidates = {
    "UGA": "Uganda",
    "RWA": "Rwanda",
    "NGA": "Nigeria",
    "KEN": "Kenya",
    "GHA": "Ghana",
    "IND": "India",
    "ZMB": "Zambia",
    "TZA": "Tanzania",
    "BGD": "Bangladesh"
}

upper_middle_candidates = {
    "IDN": "Indonesia",
    "MYS": "Malaysia",
    "BRA": "Brazil",
    "COL": "Colombia",
    "MEX": "Mexico",
    "ARG": "Argentina",
    "THA": "Thailand",
    "PHL": "Philippines",
    "MUS": "Mauritius",
    "ZAF": "South Africa"
}

all_candidates = {**lower_income_candidates, **upper_middle_candidates}
country_codes = ";".join(all_candidates.keys())

print("Candidate countries:", len(all_candidates))

Candidate countries: 19


In [4]:
# World Bank indicators used

indicators = {
    "FB.AST.NPER.ZS": "NPL_ratio",
    "NY.GDP.MKTP.KD.ZG": "GDP_growth",
    "FP.CPI.TOTL.ZG": "Inflation",
    "FS.AST.PRVT.GD.ZS": "Private_credit_GDP",
    "FB.BNK.CAPA.ZS": "Bank_capital_assets"
}


def get_world_bank_indicator(
    indicator_code,
    variable_name,
    start_year=START_YEAR,
    end_year=END_YEAR
):
    # Download one annual World Bank indicator for the candidate economies
    url = (
        "https://api.worldbank.org/v2/country/"
        f"{country_codes}/indicator/{indicator_code}"
    )

    params = {
        "format": "json",
        "date": f"{start_year}:{end_year}",
        "per_page": 20000
    }

    response = requests.get(url, params=params, timeout=60)
    response.raise_for_status()
    payload = response.json()

    if len(payload) < 2 or payload[1] is None:
        return pd.DataFrame()

    records = []

    for obs in payload[1]:
        records.append({
            "country": obs["country"]["value"],
            "country_code": obs["countryiso3code"],
            "year": int(obs["date"]),
            variable_name: obs["value"]
        })

    return pd.DataFrame(records)


# Download and merge the macro-financial series

wb_datasets = []

for indicator_code, variable_name in indicators.items():
    print(f"Downloading {variable_name}...")
    wb_datasets.append(
        get_world_bank_indicator(indicator_code, variable_name)
    )
    time.sleep(0.5)

panel = reduce(
    lambda left, right: pd.merge(
        left,
        right,
        on=["country", "country_code", "year"],
        how="outer"
    ),
    wb_datasets
)

panel = (
    panel[
        panel["country_code"].isin(all_candidates.keys())
    ]
    .sort_values(["country_code", "year"])
    .reset_index(drop=True)
)

print("World Bank panel shape:", panel.shape)

World Bank panel shape: (342, 8)


In [5]:
# Assign each country to the income group used for subgroup analysis

panel["income_group"] = np.where(panel["country_code"].isin(lower_income_candidates.keys()),"Low & Lower-Middle Income","Upper-Middle Income")

display(panel[["country", "country_code", "income_group"]].drop_duplicates().sort_values(["income_group", "country"]))

,country,country_code,income_group
18,Bangladesh,BGD,Low & Lower-Middle Income
72,Ghana,GHA,Low & Lower-Middle Income
108,India,IND,Low & Lower-Middle Income
126,Kenya,KEN,Low & Lower-Middle Income
198,Nigeria,NGA,Low & Lower-Middle Income
234,Rwanda,RWA,Low & Lower-Middle Income
270,Tanzania,TZA,Low & Lower-Middle Income
288,Uganda,UGA,Low & Lower-Middle Income
324,Zambia,ZMB,Low & Lower-Middle Income
0,Argentina,ARG,Upper-Middle Income


In [6]:
# Download annual national precipitation and temperature from the World Bank Climate Change Knowledge Portal (CRU TS 4.07)

def get_cckp_variable(
    variable_code,
    start_year=CLIMATE_HISTORY_START,
    end_year=CRU_END_YEAR,
    statistic="mean"
):
    url = (
        "https://cckpapi.worldbank.org/cckp/v1/"
        f"cru-x0.5_timeseries_{variable_code}_timeseries_annual_"
        f"{start_year}-{end_year}_{statistic}_historical_"
        f"cru_ts4.07_{statistic}/all_countries"
    )

    response = requests.get(
        url,
        params={"_format": "json"},
        timeout=120
    )
    response.raise_for_status()

    payload = response.json()["data"]
    records = []

    for iso3, yearly_values in payload.items():
        for key, value in yearly_values.items():
            records.append({
                "country_code": iso3,
                "year": int(key.split("-")[0]),
                variable_code: value
            })

    return pd.DataFrame(records)


print("Downloading precipitation...")
precip_all = get_cckp_variable("pr")

time.sleep(1)

print("Downloading temperature...")
temp_all = get_cckp_variable("tas")

climate_all = pd.merge(
    precip_all,
    temp_all,
    on=["country_code", "year"],
    how="outer"
)

climate_all = climate_all[
    climate_all["country_code"].isin(all_candidates.keys())
].copy()

print(
    "Climate coverage:",
    climate_all["year"].min(),
    "-",
    climate_all["year"].max()
)

Climate coverage: 1901 - 2022


In [7]:
# Construct country-specific climate normals over 1991–2020

climate_normal = (
    climate_all[
        climate_all["year"].between(NORMAL_START, NORMAL_END)
    ]
    .groupby("country_code")
    .agg(
        precip_normal_mean=("pr", "mean"),
        precip_normal_std=("pr", "std"),
        temp_normal_mean=("tas", "mean"),
        temp_normal_std=("tas", "std")
    )
    .reset_index()
)

climate_all = climate_all.merge(
    climate_normal,
    on="country_code",
    how="left"
)

# Standardised anomalies show how unusual each year's conditions are relative to the country's 1991–2020 climate distribution

climate_all["Precip_z"] = (
    climate_all["pr"] - climate_all["precip_normal_mean"]
) / climate_all["precip_normal_std"]

climate_all["Temp_z"] = (
    climate_all["tas"] - climate_all["temp_normal_mean"]
) / climate_all["temp_normal_std"]

# Higher drought intensity represents drier-than-normal conditions

climate_all["Drought_Intensity"] = -climate_all["Precip_z"]

In [8]:
# Restrict climate observations to the empirical study period and merge with the macro-financial panel.

climate_study = climate_all[
    climate_all["year"].between(START_YEAR, END_YEAR)
].copy()

climate_keep = [
    "country_code",
    "year",
    "pr",
    "tas",
    "Precip_z",
    "Temp_z",
    "Drought_Intensity"
]

panel = panel.merge(
    climate_study[climate_keep],
    on=["country_code", "year"],
    how="left"
)

print(
    "Merged panel years:",
    panel["year"].min(),
    "-",
    panel["year"].max()
)

Merged panel years: 2005 - 2022



## Business-cycle measure

For each country, cyclical weakness is defined as the difference between its average GDP growth rate and its observed annual GDP growth rate.
The measure is standardised within each country. A positive value therefore represents growth that is weaker than the country's own normal growth performance.

In [9]:
# Construct the continuous business-cycle weakness measure

panel = panel.sort_values(["country_code", "year"]).copy()

country_mean_growth = (
    panel.groupby("country_code")["GDP_growth"]
    .transform("mean")
)

panel["Cycle_Weakness"] = (
    country_mean_growth - panel["GDP_growth"]
)


def within_country_z(series):
    # Standardise using the country-specific mean and standard deviation.
    standard_deviation = series.std()

    if pd.isna(standard_deviation) or standard_deviation == 0:
        return pd.Series(np.nan, index=series.index)

    return (series - series.mean()) / standard_deviation


panel["Cycle_Weakness_z"] = (
    panel.groupby("country_code")["Cycle_Weakness"]
    .transform(within_country_z)
)

# The interaction captures whether drought exposure has a stronger association with credit risk when economic activity is relatively weak.

panel["Drought_x_CycleWeakness"] = (
    panel["Drought_Intensity"]
    * panel["Cycle_Weakness_z"]
)

In [10]:
# Inspect variable availability before selecting the estimation sample.

required_variables = [
    "NPL_ratio",
    "GDP_growth",
    "Inflation",
    "Private_credit_GDP",
    "Bank_capital_assets",
    "Drought_Intensity",
    "Cycle_Weakness_z"
]

availability = (
    panel.groupby(
        ["country", "country_code", "income_group"]
    )[required_variables]
    .count()
)

display(availability)

,,,NPL_ratio,GDP_growth,Inflation,Private_credit_GDP,Bank_capital_assets,Drought_Intensity,Cycle_Weakness_z
country,country_code,income_group,,,,,,,
Argentina,ARG,Upper-Middle Income,18,18,5,17,18,18,18
Bangladesh,BGD,Low & Lower-Middle Income,12,18,18,18,12,18,18
Brazil,BRA,Upper-Middle Income,18,18,18,18,18,18,18
Colombia,COL,Upper-Middle Income,18,18,18,18,18,18,18
Ghana,GHA,Low & Lower-Middle Income,15,18,18,18,15,18,18
India,IND,Low & Lower-Middle Income,14,18,18,18,14,18,18
Indonesia,IDN,Upper-Middle Income,18,18,18,14,18,18,18
Kenya,KEN,Low & Lower-Middle Income,17,18,18,18,17,18,18
Malaysia,MYS,Upper-Middle Income,18,18,18,18,18,18,18


In [11]:
# Retain countries with enough usable observations for fixed-effects estimation

panel["complete_case"] = (
    panel[required_variables]
    .notna()
    .all(axis=1)
)

coverage = (
    panel.groupby(
        ["country", "country_code", "income_group"]
    )
    .agg(
        available_years=("year", "count"),
        NPL_observations=("NPL_ratio", "count"),
        complete_observations=("complete_case", "sum")
    )
    .reset_index()
)

MIN_COMPLETE_OBS = 10

qualifying_countries = coverage.loc[
    coverage["complete_observations"] >= MIN_COMPLETE_OBS,
    "country_code"
].tolist()

final_panel = (
    panel[
        panel["country_code"].isin(qualifying_countries)
    ]
    .sort_values(["country_code", "year"])
    .reset_index(drop=True)
)

display(
    coverage[
        coverage["country_code"].isin(qualifying_countries)
    ]
    .sort_values("complete_observations", ascending=False)
)

print("Qualifying countries:", len(qualifying_countries))

,country,country_code,income_group,available_years,NPL_observations,complete_observations
10,Mexico,MEX,Upper-Middle Income,18,18,18
3,Colombia,COL,Upper-Middle Income,18,18,18
8,Malaysia,MYS,Upper-Middle Income,18,18,18
2,Brazil,BRA,Upper-Middle Income,18,18,18
7,Kenya,KEN,Low & Lower-Middle Income,18,17,17
11,Nigeria,NGA,Low & Lower-Middle Income,18,16,16
16,Thailand,THA,Upper-Middle Income,18,17,16
4,Ghana,GHA,Low & Lower-Middle Income,18,15,15
13,Rwanda,RWA,Low & Lower-Middle Income,18,15,15
14,South Africa,ZAF,Upper-Middle Income,18,15,15


Qualifying countries: 18



## Delayed NPL outcomes

Climate-related borrower stress may take time to appear in reported non-performing loans. The analysis therefore evaluates the NPL ratio contemporaneously and at horizons of one to five years.
A shifted NPL value is retained only when the corresponding observation is exactly the required number of calendar years ahead. This prevents gaps in an unbalanced panel from being incorrectly treated as consecutive years.

In [12]:
# Create contemporaneous and forward NPL outcomes

MAX_HORIZON = 5

for horizon in range(MAX_HORIZON + 1):
    outcome = f"NPL_tplus{horizon}"

    if horizon == 0:
        final_panel[outcome] = final_panel["NPL_ratio"]

    else:
        final_panel[outcome] = (
            final_panel.groupby("country_code")["NPL_ratio"]
            .shift(-horizon)
        )

        future_year = (
            final_panel.groupby("country_code")["year"]
            .shift(-horizon)
        )

        final_panel.loc[
            future_year != final_panel["year"] + horizon,
            outcome
        ] = np.nan

display(
    final_panel[
        [f"NPL_tplus{h}" for h in range(MAX_HORIZON + 1)]
    ]
    .notna()
    .sum()
    .to_frame("Available observations")
)

,Available observations
NPL_tplus0,286
NPL_tplus1,279
NPL_tplus2,271
NPL_tplus3,261
NPL_tplus4,247
NPL_tplus5,232



## Fixed-effects model

For each horizon \(h\), the model is:

NPL(i, t+h) = α + β₁ Drought Intensity(i,t) + β₂ Cycle Weakness(i,t) + β₃ [Drought Intensity(i,t) × Cycle Weakness(i,t)] + γX(i,t) + μᵢ + λₜ + ε(i,t)

The control vector contains inflation, private credit as a share of GDP, and bank capital relative to assets. Country fixed effects absorb time-invariant country characteristics, year fixed effects absorb common global shocks, and standard errors are clustered by country.

In [13]:
# Estimate the same fixed-effects specification at each NPL horizon

CONTROLS = [
    "Inflation",
    "Private_credit_GDP",
    "Bank_capital_assets"
]


def fit_amplification_model(data, dependent_variable):
    required = [
        dependent_variable,
        "Drought_Intensity",
        "Cycle_Weakness_z",
        "Drought_x_CycleWeakness",
        *CONTROLS
    ]

    estimation_data = data.dropna(subset=required).copy()

    # At least eight usable observations are required per country at each horizon because longer delays mechanically shorten the available panel
    country_counts = estimation_data.groupby("country_code").size()
    retained = country_counts[country_counts >= 8].index

    estimation_data = estimation_data[
        estimation_data["country_code"].isin(retained)
    ].copy()

    indexed_data = estimation_data.set_index(
        ["country_code", "year"]
    ).sort_index()

    formula = (
        f"{dependent_variable} ~ 1 "
        "+ Drought_Intensity "
        "+ Cycle_Weakness_z "
        "+ Drought_x_CycleWeakness "
        "+ Inflation "
        "+ Private_credit_GDP "
        "+ Bank_capital_assets "
        "+ EntityEffects "
        "+ TimeEffects"
    )

    model = PanelOLS.from_formula(
        formula,
        data=indexed_data,
        drop_absorbed=True
    )

    result = model.fit(
        cov_type="clustered",
        cluster_entity=True
    )

    return result, estimation_data

In [14]:
# Estimate the full-sample model from t through t+5

full_sample_results = {}
full_sample_rows = []

for horizon in range(MAX_HORIZON + 1):
    dependent_variable = f"NPL_tplus{horizon}"

    result, sample = fit_amplification_model(
        final_panel,
        dependent_variable
    )

    full_sample_results[horizon] = result

    term = "Drought_x_CycleWeakness"

    full_sample_rows.append({
        "Horizon": horizon,
        "Outcome": "NPL_t" if horizon == 0 else f"NPL_t+{horizon}",
        "Interaction_coefficient": result.params.get(term, np.nan),
        "Standard_error": result.std_errors.get(term, np.nan),
        "P_value": result.pvalues.get(term, np.nan),
        "Within_R2": result.rsquared_within,
        "Observations": result.nobs,
        "Countries": sample["country_code"].nunique()
    })

full_sample_table = pd.DataFrame(full_sample_rows)

# Holm-adjusted p-values account for testing the interaction at several horizons

valid_p = full_sample_table["P_value"].notna()

full_sample_table.loc[valid_p, "Holm_adjusted_P"] = multipletests(
    full_sample_table.loc[valid_p, "P_value"],
    method="holm"
)[1]

display(full_sample_table.round(4))

,Horizon,Outcome,Interaction_coefficient,Standard_error,P_value,Within_R2,Observations,Countries,Holm_adjusted_P
0,0,NPL_t,-0.0141,0.1324,0.9154,0.0452,271,18,1.0000
1,1,NPL_t+1,0.1255,0.1227,0.3076,0.0463,251,18,1.0000
2,2,NPL_t+2,0.2443,0.1470,0.0980,0.1738,233,18,0.5881
3,3,NPL_t+3,-0.0249,0.1613,0.8774,0.0949,217,18,1.0000
4,4,NPL_t+4,-0.1249,0.2087,0.5504,-0.0362,198,18,1.0000
5,5,NPL_t+5,-0.1470,0.2184,0.5023,0.0091,159,15,1.0000


In [15]:
# Display the full regression summary for every horizon

for horizon, result in full_sample_results.items():
    label = "NPL_t" if horizon == 0 else f"NPL_t+{horizon}"

    print("\n" + "=" * 80)
    print(label)
    print("=" * 80)
    print(result.summary)


NPL_t
                          PanelOLS Estimation Summary                           
Dep. Variable:             NPL_tplus0   R-squared:                        0.0822
Estimator:                   PanelOLS   R-squared (Between):             -0.4944
No. Observations:                 271   R-squared (Within):               0.0452
Date:                Wed, Sep 23 2026   R-squared (Overall):             -0.2684
Time:                        18:57:49   Log-likelihood                   -617.23
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      3.4342
Entities:                          18   P-value                           0.0029
Avg Obs:                       15.056   Distribution:                   F(6,230)
Min Obs:                       12.000                                           
Max Obs:                       18.000   F-statistic (robust):             2.5597
                     


## Income-group analysis

The same model is estimated separately for low/lower-middle-income and upper-middle-income economies at each horizon. These regressions describe the pattern within each group.
A pooled triple-interaction model is then used to test formally whether the drought–cycle amplification coefficient differs between the two income groups.

In [16]:
# Estimate the interaction separately within each income group

income_group_rows = []

for horizon in range(MAX_HORIZON + 1):
    dependent_variable = f"NPL_tplus{horizon}"

    for group_label in [
        "Low & Lower-Middle Income",
        "Upper-Middle Income"
    ]:
        group_data = final_panel[
            final_panel["income_group"] == group_label
        ].copy()

        try:
            result, sample = fit_amplification_model(
                group_data,
                dependent_variable
            )

            term = "Drought_x_CycleWeakness"

            income_group_rows.append({
                "Horizon": horizon,
                "Income_group": group_label,
                "Interaction_coefficient": result.params.get(term, np.nan),
                "Standard_error": result.std_errors.get(term, np.nan),
                "P_value": result.pvalues.get(term, np.nan),
                "Within_R2": result.rsquared_within,
                "Observations": result.nobs,
                "Countries": sample["country_code"].nunique()
            })

        except Exception:
            income_group_rows.append({
                "Horizon": horizon,
                "Income_group": group_label,
                "Interaction_coefficient": np.nan,
                "Standard_error": np.nan,
                "P_value": np.nan,
                "Within_R2": np.nan,
                "Observations": np.nan,
                "Countries": np.nan
            })

income_group_table = pd.DataFrame(income_group_rows)

display(income_group_table.round(4))

,Horizon,Income_group,Interaction_coefficient,Standard_error,P_value,Within_R2,Observations,Countries
0,0,Low & Lower-Middle Income,-0.1692,0.3761,0.6538,0.1117,126,9
1,0,Upper-Middle Income,-0.0143,0.0564,0.8001,0.0999,145,9
2,1,Low & Lower-Middle Income,0.2219,0.2396,0.3569,0.1261,115,9
3,1,Upper-Middle Income,-0.0647,0.0583,0.2702,0.0017,136,9
4,2,Low & Lower-Middle Income,0.7692,0.2897,0.0096,0.3370,106,9
5,2,Upper-Middle Income,-0.1150,0.0541,0.0362,-0.0644,127,9
6,3,Low & Lower-Middle Income,-0.1809,0.4342,0.6783,0.0674,99,9
7,3,Upper-Middle Income,0.0044,0.0185,0.8121,-0.2414,118,9
8,4,Low & Lower-Middle Income,-0.5734,0.3284,0.0858,-0.0403,89,9
9,4,Upper-Middle Income,0.0096,0.1046,0.9273,-0.1738,109,9


In [17]:
# Formal test of whether amplification differs between income groups

def fit_income_difference_model(data, dependent_variable):
    model_data = data.copy()

    model_data["LowLowerMiddle"] = (
        model_data["income_group"] == "Low & Lower-Middle Income"
    ).astype(int)

    model_data["Drought_x_Low"] = (
        model_data["Drought_Intensity"]
        * model_data["LowLowerMiddle"]
    )

    model_data["CycleWeakness_x_Low"] = (
        model_data["Cycle_Weakness_z"]
        * model_data["LowLowerMiddle"]
    )

    model_data["Drought_x_CycleWeakness_x_Low"] = (
        model_data["Drought_Intensity"]
        * model_data["Cycle_Weakness_z"]
        * model_data["LowLowerMiddle"]
    )

    required = [
        dependent_variable,
        "Drought_Intensity",
        "Cycle_Weakness_z",
        "Drought_x_CycleWeakness",
        "Drought_x_Low",
        "CycleWeakness_x_Low",
        "Drought_x_CycleWeakness_x_Low",
        *CONTROLS
    ]

    model_data = model_data.dropna(subset=required).copy()

    country_counts = model_data.groupby("country_code").size()
    retained = country_counts[country_counts >= 8].index

    model_data = model_data[
        model_data["country_code"].isin(retained)
    ].copy()

    indexed_data = model_data.set_index(
        ["country_code", "year"]
    ).sort_index()

    formula = (
        f"{dependent_variable} ~ 1 "
        "+ Drought_Intensity "
        "+ Cycle_Weakness_z "
        "+ Drought_x_CycleWeakness "
        "+ Drought_x_Low "
        "+ CycleWeakness_x_Low "
        "+ Drought_x_CycleWeakness_x_Low "
        "+ Inflation "
        "+ Private_credit_GDP "
        "+ Bank_capital_assets "
        "+ EntityEffects "
        "+ TimeEffects"
    )

    result = PanelOLS.from_formula(
        formula,
        data=indexed_data,
        drop_absorbed=True
    ).fit(
        cov_type="clustered",
        cluster_entity=True
    )

    return result, model_data

In [18]:
# Evaluate the formal income-group difference at each horizon

income_difference_results = {}
income_difference_rows = []

for horizon in range(MAX_HORIZON + 1):
    dependent_variable = f"NPL_tplus{horizon}"

    result, sample = fit_income_difference_model(
        final_panel,
        dependent_variable
    )

    income_difference_results[horizon] = result

    term = "Drought_x_CycleWeakness_x_Low"

    income_difference_rows.append({
        "Horizon": horizon,
        "Outcome": "NPL_t" if horizon == 0 else f"NPL_t+{horizon}",
        "Difference_coefficient": result.params.get(term, np.nan),
        "Standard_error": result.std_errors.get(term, np.nan),
        "P_value": result.pvalues.get(term, np.nan),
        "Within_R2": result.rsquared_within,
        "Observations": result.nobs,
        "Countries": sample["country_code"].nunique()
    })

income_difference_table = pd.DataFrame(
    income_difference_rows
)

valid_p = income_difference_table["P_value"].notna()

income_difference_table.loc[
    valid_p,
    "Holm_adjusted_P"
] = multipletests(
    income_difference_table.loc[valid_p, "P_value"],
    method="holm"
)[1]

display(income_difference_table.round(4))

,Horizon,Outcome,Difference_coefficient,Standard_error,P_value,Within_R2,Observations,Countries,Holm_adjusted_P
0,0,NPL_t,-0.1535,0.3404,0.6525,0.0587,271,18,0.7928
1,1,NPL_t+1,0.3482,0.2243,0.1220,0.0542,251,18,0.4879
2,2,NPL_t+2,0.7104,0.3150,0.0253,0.2036,233,18,0.1515
3,3,NPL_t+3,-0.3277,0.3855,0.3964,0.0984,217,18,0.7928
4,4,NPL_t+4,-0.8721,0.4566,0.0580,-0.0193,198,18,0.2898
5,5,NPL_t+5,-0.8842,0.6013,0.1440,0.0259,159,15,0.4879


In [19]:
# Compact result tables for final interpretation

print("FULL-SAMPLE LAG PROFILE")
display(
    full_sample_table[
        [
            "Horizon",
            "Outcome",
            "Interaction_coefficient",
            "Standard_error",
            "P_value",
            "Holm_adjusted_P",
            "Within_R2",
            "Observations",
            "Countries"
        ]
    ].round(4)
)

print("\nINCOME-GROUP LAG PROFILE")
display(
    income_group_table.round(4)
)

print("\nFORMAL INCOME-GROUP DIFFERENCE")
display(
    income_difference_table.round(4)
)

FULL-SAMPLE LAG PROFILE


,Horizon,Outcome,Interaction_coefficient,Standard_error,P_value,Holm_adjusted_P,Within_R2,Observations,Countries
0,0,NPL_t,-0.0141,0.1324,0.9154,1.0000,0.0452,271,18
1,1,NPL_t+1,0.1255,0.1227,0.3076,1.0000,0.0463,251,18
2,2,NPL_t+2,0.2443,0.1470,0.0980,0.5881,0.1738,233,18
3,3,NPL_t+3,-0.0249,0.1613,0.8774,1.0000,0.0949,217,18
4,4,NPL_t+4,-0.1249,0.2087,0.5504,1.0000,-0.0362,198,18
5,5,NPL_t+5,-0.1470,0.2184,0.5023,1.0000,0.0091,159,15



INCOME-GROUP LAG PROFILE


,Horizon,Income_group,Interaction_coefficient,Standard_error,P_value,Within_R2,Observations,Countries
0,0,Low & Lower-Middle Income,-0.1692,0.3761,0.6538,0.1117,126,9
1,0,Upper-Middle Income,-0.0143,0.0564,0.8001,0.0999,145,9
2,1,Low & Lower-Middle Income,0.2219,0.2396,0.3569,0.1261,115,9
3,1,Upper-Middle Income,-0.0647,0.0583,0.2702,0.0017,136,9
4,2,Low & Lower-Middle Income,0.7692,0.2897,0.0096,0.3370,106,9
5,2,Upper-Middle Income,-0.1150,0.0541,0.0362,-0.0644,127,9
6,3,Low & Lower-Middle Income,-0.1809,0.4342,0.6783,0.0674,99,9
7,3,Upper-Middle Income,0.0044,0.0185,0.8121,-0.2414,118,9
8,4,Low & Lower-Middle Income,-0.5734,0.3284,0.0858,-0.0403,89,9
9,4,Upper-Middle Income,0.0096,0.1046,0.9273,-0.1738,109,9



FORMAL INCOME-GROUP DIFFERENCE


,Horizon,Outcome,Difference_coefficient,Standard_error,P_value,Within_R2,Observations,Countries,Holm_adjusted_P
0,0,NPL_t,-0.1535,0.3404,0.6525,0.0587,271,18,0.7928
1,1,NPL_t+1,0.3482,0.2243,0.1220,0.0542,251,18,0.4879
2,2,NPL_t+2,0.7104,0.3150,0.0253,0.2036,233,18,0.1515
3,3,NPL_t+3,-0.3277,0.3855,0.3964,0.0984,217,18,0.7928
4,4,NPL_t+4,-0.8721,0.4566,0.0580,-0.0193,198,18,0.2898
5,5,NPL_t+5,-0.8842,0.6013,0.1440,0.0259,159,15,0.4879
